In [ ]:
!pip install -q torch scikit-learn numpy matplotlib netcal

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from netcal.metrics import ECE
from google.colab import drive

drive.mount('/content/drive')

SAVE_PATH = '/content/drive/MyDrive/ptb-xl-dataset/'
CKPT_PATH = '/content/drive/MyDrive/ptb-xl-dataset/'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

X_train_norm = np.load(SAVE_PATH + 'X_train_norm.npy')
X_test_norm  = np.load(SAVE_PATH + 'X_test_norm.npy')
y_train_enc  = np.load(SAVE_PATH + 'y_train_enc.npy')
y_test_enc   = np.load(SAVE_PATH + 'y_test_enc.npy')

# Fold-9 split (same as notebooks 04 and 05)
Y = pd.read_csv(SAVE_PATH + 'ptb-xl/ptbxl_database.csv', index_col='ecg_id')
train_meta = Y[Y.strat_fold != 10]
val_mask   = (train_meta.strat_fold == 9).values
train_mask = (train_meta.strat_fold != 9).values
X_train, y_train = X_train_norm[train_mask], y_train_enc[train_mask]
X_val,   y_val   = X_train_norm[val_mask],   y_train_enc[val_mask]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test_norm.shape} | Device: {device}")


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7, padding=3, bias=False)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.skip  = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.skip(x)
        return F.relu(out)


class ECGEncoder(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()
        self.conv1   = nn.Conv1d(12, 64, kernel_size=15, stride=2, padding=7, bias=False)
        self.bn1     = nn.BatchNorm1d(64)
        self.layer1  = ResidualBlock(64,  64)
        self.layer2  = ResidualBlock(64,  128, stride=2)
        self.layer3  = ResidualBlock(128, 256, stride=2)
        self.layer4  = ResidualBlock(256, 512, stride=2)
        self.pool    = nn.AdaptiveAvgPool1d(1)
        self.project = nn.Linear(512, embedding_dim)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.pool(x).squeeze(-1)
        return self.project(x)


class ECGDataset(Dataset):
    def __init__(self, signals, labels):
        self.signals = torch.FloatTensor(signals)
        self.labels  = torch.FloatTensor(labels)
    def __len__(self): return len(self.signals)
    def __getitem__(self, idx): return self.signals[idx], self.labels[idx]


class TemperatureScaling(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))
    def forward(self, logits):
        return torch.sigmoid(logits / self.temperature)

print("Setup done.")

Mounted at /content/drive
Train: (17418, 1000, 12) | Val: (2183, 1000, 12) | Test: (2198, 1000, 12) | Device: cpu
Setup done.


In [ ]:
# Re-seed, sample 10% subset, fine-tune CPC (matches notebook 05)
random.seed(42); np.random.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)

n_samples = int(len(X_train) * 0.10)
indices   = np.random.choice(len(X_train), n_samples, replace=False)
X_tr_10, y_tr_10 = X_train[indices], y_train[indices]

cpc_encoder = ECGEncoder(256).to(device)
cpc_encoder.load_state_dict(torch.load(CKPT_PATH + 'cpc_checkpoint_epoch50.pt', map_location=device)['encoder_state'])
cpc_model = nn.Sequential(cpc_encoder, nn.Linear(256, 5), nn.Sigmoid()).to(device)

g = torch.Generator(); g.manual_seed(42)
loader    = DataLoader(ECGDataset(X_tr_10, y_tr_10), batch_size=64, shuffle=True, generator=g)
optimizer = optim.Adam(cpc_model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.BCELoss()

cpc_model.train()
for epoch in range(20):
    ep = 0.0
    for sig, lab in loader:
        sig, lab = sig.to(device), lab.to(device)
        loss = criterion(cpc_model(sig), lab)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        ep += loss.item()
    print(f"Epoch {epoch+1:02d}/20 — Loss: {ep/len(loader):.4f}")
print("CPC fine-tuning complete.")


def get_logits(model, X):
    enc, lin = model[0], model[1]
    enc.eval(); lin.eval()
    loader = DataLoader(ECGDataset(X, np.zeros((len(X), 5))), batch_size=64, shuffle=False)
    out = []
    with torch.no_grad():
        for sig, _ in loader:
            out.append(lin(enc(sig.to(device))).cpu())
    return torch.cat(out)

def fit_temperature(model, Xv, yv):
    logits = get_logits(model, Xv).to(device)
    labels = torch.FloatTensor(yv).to(device)
    tm = TemperatureScaling().to(device)
    opt = optim.LBFGS([tm.temperature], lr=0.01, max_iter=50)
    crit = nn.BCELoss()
    def closure():
        opt.zero_grad(); l = crit(tm(logits), labels); l.backward(); return l
    opt.step(closure)
    print(f"Optimal temperature: {tm.temperature.item():.4f}")
    return tm

cpc_temp = fit_temperature(cpc_model, X_val, y_val)


def compute_ece_per_class(probs, labels, n_bins=10):
    m = ECE(n_bins)
    return float(np.mean([m.measure(probs[:, c], labels[:, c]) for c in range(labels.shape[1])]))

def mask_leads(X, lead_indices):
    Xm = np.zeros_like(X)
    Xm[:, :, lead_indices] = X[:, :, lead_indices]
    return Xm

def evaluate_lead_setting(model, temp, X_test, y_test, lead_indices, name):
    Xm = mask_leads(X_test, lead_indices)
    logits = get_logits(model, Xm).to(device)
    with torch.no_grad():
        probs = temp(logits).cpu().numpy()
    preds = (probs > 0.5).astype(int)
    auc   = roc_auc_score(y_test, probs, average='macro')
    f1    = f1_score(y_test, preds, average='macro')
    brier = np.mean([brier_score_loss(y_test[:, i], probs[:, i]) for i in range(5)])
    eps   = 1e-7
    nll   = -np.mean(y_test*np.log(probs+eps) + (1-y_test)*np.log(1-probs+eps))
    ece   = compute_ece_per_class(probs, y_test)   # per-class ECE (fixed)
    print(f"{name:10s} | AUC: {auc:.4f} | F1: {f1:.4f} | Brier: {brier:.4f} | NLL: {nll:.4f} | ECE: {ece:.4f}")
    return auc, f1, brier, nll, ece

lead_configs = {
    '12-lead': list(range(12)),
    '6-lead':  [0, 1, 2, 5, 6, 10],
    '1-lead':  [0],
}

print("\nReduced-Lead Evaluation — Calibrated CPC Model")
print("=" * 70)
results = {}
for name, idx in lead_configs.items():
    results[name] = evaluate_lead_setting(cpc_model, cpc_temp, X_test_norm, y_test_enc, idx, name)

Epoch 01/20 — Loss: 0.4471
Epoch 02/20 — Loss: 0.3204
Epoch 03/20 — Loss: 0.2870
Epoch 04/20 — Loss: 0.2667
Epoch 05/20 — Loss: 0.2453
Epoch 06/20 — Loss: 0.2237
Epoch 07/20 — Loss: 0.2132
Epoch 08/20 — Loss: 0.1913
Epoch 09/20 — Loss: 0.1829
Epoch 10/20 — Loss: 0.1638
Epoch 11/20 — Loss: 0.1444
Epoch 12/20 — Loss: 0.1345
Epoch 13/20 — Loss: 0.1230
Epoch 14/20 — Loss: 0.1031
Epoch 15/20 — Loss: 0.0764
Epoch 16/20 — Loss: 0.0643
Epoch 17/20 — Loss: 0.0596
Epoch 18/20 — Loss: 0.0504
Epoch 19/20 — Loss: 0.0649
Epoch 20/20 — Loss: 0.0860
CPC fine-tuning complete.
Optimal temperature: 1.2484

Reduced-Lead Evaluation — Calibrated CPC Model
12-lead    | AUC: 0.8381 | F1: 0.5248 | Brier: 0.1875 | NLL: 0.7533 | ECE: 0.1863
6-lead     | AUC: 0.7248 | F1: 0.3142 | Brier: 0.3052 | NLL: 1.1534 | ECE: 0.2903
1-lead     | AUC: 0.5280 | F1: 0.2219 | Brier: 0.3359 | NLL: 1.0574 | ECE: 0.3373
